## Image Representations

<h2><b>What is the meaning of 'representations of images'?</b></h2>

An image representation refers to the process of translating raw, high-dimensional visual data (like tens of thousands of RGB pixel values) into a compressed, abstract, or more useful mathematical form.
Fundamentally, image representation is rooted in dimensionality reduction. While raw image data $x$ exists in a massive, high-dimensional coordinate system, realistic images (like faces or natural scenes) only occupy a tiny subset or "manifold" of that space. Creating a representation means finding a "low-dimensional (or hidden) representation $h$, which can approximately explain the data $x$, so that $x \approx f[h,\theta]$".

<h3>What is achieved by image representation?</h3>

1. <i>Dimensionality reduction</i> : From thousands of rows of RGB values, we come down to few hundreds of rows of representation.
2. <i> A Compact Summary of Critical Information</i>: A representation serves as a "highly compact representation" that summarizes an image region while retaining the "critical information about the image appearance and layout".
3. <i> A Method to Filter Out Irrelevant Variations </i>: In real-world computer vision, raw RGB values constantly fluctuate due to factors that have nothing to do with the actual object, such as changes in ambient lighting, camera gain, or slight shifts in pose.
4. <i> An Intermediate Stepping Stone for Inference </i>: Representations act as an "intermediate representation for inferring higher-level properties".

<h2><b>The Dataset: ImageNet</b></h2>

**ImageNet** is a massive, foundational visual database designed specifically to support visual object recognition and computer vision research. It serves as a comprehensive "visual dictionary" for computers and has been instrumental in advancing deep learning models.

* **Scale and Scope:** The full dataset (sometimes referred to as ImageNet-21K) contains more than 14 million images that have been hand-annotated.
* **Organization:** It is structured according to the WordNet lexical hierarchy, where each node or concept (called a "synset") is depicted by hundreds or thousands of images. The full dataset contains over 20,000 distinct categories, ranging from specific dog breeds to everyday objects like balloons or strawberries.
* **Annotation:** The images were scraped from search engines and then rigorously human-annotated using Amazon's Mechanical Turk platform to verify the presence of specific objects. Over one million of these images also include object-level bounding boxes to show exactly where the object is located.

Working on entire ImageNet dataset is both laborious and unnecessary. Therefore, this study of creating representations is done on a subset of ImageNet. A 100,000 images were sampled at random from the dataset and were subsequently split into training, validation and test sets in the ratio of 8:1:1 respectively.

<h2><b>The Architecture: Convolutional Autoencoder</b></h2>

The ImageNet dataset is vast and it varies greatly. Hence the ideal architecture to encode the image is a **Convolutional Autoencoder** where stacks of <code>nn.Conv2d</code> and <code>nn.MaxPool2d</code> layers. Each step will increase the number of feature channels while slicing the spatial dimensions ($H \times W$) in half, squeezing the image down into your dense, low-dimensional latent representation.

The same network can later function as a decoder to extract images from the representations.

<h3><b>The Pipeline</b></h3>

* **1. Dataset:** 100k random ImageNet images, split 8:1:1 (80k Train, 10k Val, 10k Test).
* **2. Pre-processing:** Resize all images to a uniform resolution (e.g., $64 \times 64$), convert to PyTorch tensors, and normalize the RGB channels.
* **3. Architecture:** An hourglass design. The **Encoder** (`Conv2d` + `MaxPool`) compresses images into a dense latent representation. The **Decoder** (`ConvTranspose2d`) reconstructs the representation back into pixels.
* **4. Training & Validation:** Optimize using **MSE Loss** (measuring pixel reconstruction error). Monitor validation loss after each epoch to trigger **Early Stopping** and prevent overfitting.



## <b>Data Loading and pre-processing</b>


In [13]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# setting up seed
seed = 42
torch.manual_seed(seed=seed)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(42)

# 4. Force PyTorch to use deterministic algorithms
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Random seeds locked to {seed}. Training is now reproducible.")

Random seeds locked to 42. Training is now reproducible.


In [14]:
# set up the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Subset, dataloader

In [15]:
# 1. define the transformation
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Load the Tiny ImageNet dataset from Hugging Face
from datasets import load_dataset
ds = load_dataset("zh-plus/tiny-imagenet")

# 3. Simple random splits
splits = ds["train"].train_test_split(test_size=20000, seed=42)
train_dataset = splits["train"]  # 80,000 random images

val_test_splits = splits["test"].train_test_split(test_size=10000, seed=42)
val_dataset = val_test_splits["train"]  # 10,000 random images
test_dataset = val_test_splits["test"]   # Remaining 10,000 random images

# 4. Define the preprocessing transform function to prepare 'pixel_values'
def preprocess(examples):
    examples["pixel_values"] = [transform(img.convert("RGB")) for img in examples["image"]]
    return examples

# Apply the transform dynamically to each split
train_dataset.set_transform(preprocess)
val_dataset.set_transform(preprocess)
test_dataset.set_transform(preprocess)

# Remove the original 'image' column to prevent DataLoader from trying to batch PIL images
train_dataset = train_dataset.remove_columns(["image"])
val_dataset = val_dataset.remove_columns(["image"])
test_dataset = test_dataset.remove_columns(["image"])

# 5. Create PyTorch DataLoaders
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 6. Verify sizes
print(f"Data loading complete:")
print(f" - Train samples: {len(train_dataset)}")
print(f" - Val samples:   {len(val_dataset)}")
print(f" - Test samples:  {len(test_dataset)}")


Data loading complete:
 - Train samples: 80000
 - Val samples:   10000
 - Test samples:  10000


In [16]:
import random

# Set the seed for reproducibility
random.seed(seed)

# 1. Get 5 random indices from the training dataset
random_indices = random.sample(range(len(train_dataset)), 5)

# 2. Denormalization parameters (matching the transform's mean and std)
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# 3. Plot the de-normalized images
plt.figure(figsize=(15, 5))
for i, idx in enumerate(random_indices):
    # This retrieves the preprocessed sample (dict containing 'pixel_values' tensor and 'label')
    sample = train_dataset[idx]
    img = sample['pixel_values'].cpu()

    # De-normalize the tensor back to the original pixel range [0, 1]
    img = img * std + mean

    # Convert shape from (C, H, W) to (H, W, C) for Matplotlib
    img = img.numpy().transpose((1, 2, 0))
    img = np.clip(img, 0, 1) # Clip values to keep them in valid [0, 1] range

    label = sample['label']

    plt.subplot(1, 5, i + 1)
    plt.imshow(img)
    plt.title(f"Idx: {idx}\nLabel: {label}")
    plt.axis('off')

plt.tight_layout()
plt.show()


KeyError: 'image'

<Figure size 1500x500 with 0 Axes>

## <h2><b>The Architecture</b></h2>

In [ ]:
class CNNAutoencoder(nn.Module):
    def __init__(self):
        super(CNNAutoencoder, self).__init__()
        # ====================================
        # THE ENCODER
        # Input shape: [Batch, 3, 64, 64]
        # ====================================

        self.encoder = nn.Sequential(
            # Layer 1: 3 channels -> 16 channels, size halves to 32 x 32
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),

            # Layer 2: 16 channels -> 32 channels, size halves to 16 x 16
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),

            # Layer 3: 32 channels -> 64 channels, size halves to 8 x 8
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),

            # Layer 4: 64 channels -> 128 channels, size halves to 4 x 4
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True)
        )
        # now the output representation will be a tensor of 128x4x4

        # ======================================
        # THE DECODER
        # Input shape: [Batch, 128, 4, 4]
        # ======================================

        self.decoder = nn.Sequential(
            # Layer 1: 128 channels -> 64 channels, size doubles to 8 x 8
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(True),

            # Layer 2: 64 channels -> 32 channel, size doubles to 16 x 16
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(True),

            # Layer 3: 32 channels -> 16 channels, size doubles to 32 x 32
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(True),

            # Layer 4: 16 channels -> 3 channels, size doubles to 64 x 64
            nn.ConvTranspose2d(16, 3, kernel_size=3, stride=2, padding=1, output_padding=1),

        )
        # now the output representation will be a tensor of 3x64x64 like the original image

        # Note: Because the images were normalized before passing through the encoder, a linear
        # transformation is enough and there is no need of nn.Sigmoid or tanh activations
    def forward(self, x):
        representation = self.encoder(x)
        reconstruction = self.decoder(representation)
        return reconstruction

    def get_representation(self, x):
        # A handy helper method for your downstream task!
        return self.encoder(x)

## <b>Training the Autoencoder</b>

### <i>Hyperparameter Tuning </i>

In [ ]:
!pip install optuna

In [ ]:
import optuna
tune_train_ds = Subset(train_dataset, range(5000))
tune_val_ds = Subset(val_dataset, range(500))
# making format torch

tune_train_loader = DataLoader(tune_train_ds, batch_size=64, shuffle=True)
tune_val_loader = DataLoader(tune_val_ds, batch_size=64, shuffle=False)

def objective(trial):
  model = CNNAutoencoder().to(device)

  # optimizer choices
  optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"])

  # learning rate choices
  lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)

  # weight decay choices
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-1, log=True)

  # dynamically fetch the chosen optimizer from torch.optim
  optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr, weight_decay=weight_decay)

  # MSE loss
  criterion = nn.MSELoss()

  epochs = 5
  for epoch in range(epochs):
    # training phase
    model.train()
    for batch_idx, data in enumerate(tune_train_loader):
    # --- The Crucial Extraction Step ---
    # If data is a dictionary (HuggingFace standard)
      if isinstance(data, dict):
          # Use 'image' or 'pixel_values' depending on your transform function
          images = data['pixel_values'].to(device)
      # If data is a list/tuple (PyTorch standard)
      else:
          images = data[0].to(device)


      optimizer.zero_grad()

      # Forward pass
      reconstruction = model(images)

      # Calculate MSE loss (comparing generated image to original image)
      loss = criterion(reconstruction, images)

      # Backprop
      loss.backward()
      optimizer.step()

    # validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
            for data in tune_val_loader:
                images = data[0].to(device) if isinstance(data, list) else data['pixel_values'].to(device)
                reconstruction = model(images)
                loss = criterion(reconstruction, images)
                val_loss += loss.item() * images.size(0)

    val_loss /= len(tune_val_loader.dataset)

    # Report back to Optuna to allow for Pruning (stopping bad trials early)
    trial.report(val_loss, epoch)
    if trial.should_prune():
        raise optuna.exceptions.TrialPruned()

  # Optuna will try to minimize this returned value
  return val_loss

# --- RUN THE STUDY ---
print("Starting Hyperparameter Tuning...")
study = optuna.create_study(direction="minimize")
# Run for 20 trials to see what works best
study.optimize(objective, n_trials=20)

print(f"Best trial parameters: {study.best_params}")
